<a href="https://colab.research.google.com/github/jihanbakshi7/Explainable-Multi-modal-Fact-Verification-System-Using-Context-Aware-Retrieval-Augmented-Evi.-Gen./blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOTEBOOK 1 / 6 — Data Prep (fresh, n=20 fast test set)
**Runtime: CPU is fine.** No GPU needed.

Downloads AVeriTeC, samples **20 test claims** (stratified across all 4
labels, for a quick full-pipeline test), scrapes their evidence pages, and
chunks them. Only annotated answer-source URLs are collected, preventing
fact-check article leakage. Output: `test_df.parquet`, `chunks_df.parquet`,
`evidence_docs_df.parquet`, and `claim_sources_df.parquet`, saved to a
versioned shared Drive checkpoint folder.


In [ ]:
# ===== Cell 1: Install — CPU =====
!pip -q install -U pip
!pip -q install pandas numpy tqdm requests beautifulsoup4 lxml trafilatura readability-lxml pyarrow
print('Dependencies installed.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
# ===== Cell 2: Mount Drive + shared checkpoint folder — CPU =====
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

RUN_NAME = 'averitec_20claim_v1'
PROJECT_DIR = Path('/content/drive/MyDrive/averitec_extended')
CHECKPOINT_DIR = PROJECT_DIR / RUN_NAME / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint folder (shared by all notebooks):', CHECKPOINT_DIR)


Mounted at /content/drive
Checkpoint folder (shared by all notebooks): /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints


In [ ]:
# ===== Cell 3: Imports + settings — CPU =====
import os, re, json, time, random, hashlib, math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Fast test settings: n=20 claims, stratified across all 4 labels ──
USE_FULL_DEV_SET = False   # False = 20-claim stratified sample (fast); True = full ~500-claim dev set

N_EVIDENCE_URLS = 150       # generous for 20 claims; raise if scaling up later

LABEL_SAMPLE = {
    'Refuted'             : 8,
    'Supported'           : 6,
    'Not Enough Evidence' : 4,
    'Conflicting Evidence': 2,
}   # totals 20

# ── Chunking settings ──
CHUNK_WORDS     = 180
OVERLAP_WORDS   = 40
MIN_CHUNK_WORDS = 30

# ── AVeriTeC dataset ──
AVERITEC_DEV_URL = 'https://raw.githubusercontent.com/MichSchli/AVeriTeC/main/data/dev.json'

# ── HTTP settings ──
HTTP_TIMEOUT   = 15
MIN_WORD_COUNT = 80
SLEEP_SECONDS  = 0.3
USER_AGENT     = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'

LABELS = ['Supported', 'Refuted', 'Not Enough Evidence', 'Conflicting Evidence']

print('Settings ready. Test set size:', 'FULL DEV SET (~500)' if USE_FULL_DEV_SET else '20 claims (fast test)')


Settings ready. Test set size: 20 claims (fast test)


In [ ]:
# ===== Cell 4: Utility functions — CPU =====

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ''
    x = str(x)
    x = x.replace('\xa0', ' ')
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def normalize_label(text):
    if not isinstance(text, str):
        return 'Unknown'
    t = text.lower().strip()
    if any(k in t for k in ['not enough', 'insufficient', 'cannot determine', 'unknown', 'unverified']):
        return 'Not Enough Evidence'
    if any(k in t for k in ['conflicting', 'mixed', 'cherry']):
        return 'Conflicting Evidence'
    if any(k in t for k in ['refuted', 'false', 'incorrect', 'contradict']):
        return 'Refuted'
    if any(k in t for k in ['supported', 'true', 'correct', 'support']):
        return 'Supported'
    return 'Unknown'


def url_to_doc_id(url):
    return hashlib.md5(url.encode('utf-8')).hexdigest()


def safe_word_count(text):
    return len(text.split()) if isinstance(text, str) else 0

print('Utility functions ready.')


Utility functions ready.


## STEP 1 — Download AVeriTeC dataset + build test set

In [ ]:
# ===== Cell 5: Download + normalize AVeriTeC =====
print('Downloading AVeriTeC dev.json...')
resp = requests.get(AVERITEC_DEV_URL, timeout=30)
resp.raise_for_status()
raw_dev = resp.json()
print(f'Dev records downloaded: {len(raw_dev)}')


def normalize_record(rec, fallback_id):
    claim_id = str(rec.get('claim_id', rec.get('id', fallback_id)))
    claim    = clean_text(rec.get('claim', rec.get('claim_text', '')))
    label    = normalize_label(str(rec.get('label', rec.get('verdict', ''))))

    # Use only annotated answer sources. Collecting every URL in the record
    # would also include the fact-checking article and leak the verdict.
    source_urls = []
    gold_qa_pairs = []
    for question_record in rec.get('questions', []):
        question = clean_text(question_record.get('question', ''))
        normalized_answers = []

        for answer_record in question_record.get('answers', []):
            if not isinstance(answer_record, dict):
                continue

            answer_text = clean_text(answer_record.get('answer', ''))
            source_url = clean_text(answer_record.get('source_url', ''))
            cached_source_url = clean_text(answer_record.get('cached_source_url', ''))

            if source_url.startswith('http'):
                source_urls.append(source_url)

            normalized_answers.append({
                'answer': answer_text,
                'source_url': source_url,
                'cached_source_url': cached_source_url,
                'answer_type': answer_record.get('answer_type', ''),
                'source_medium': answer_record.get('source_medium', ''),
            })

        gold_qa_pairs.append({
            'question': question,
            'answer': normalized_answers[0]['answer'] if normalized_answers else '',
            'answers': normalized_answers,
        })

    return {'claim_id': claim_id, 'claim': claim, 'label': label,
            'source_urls': list(dict.fromkeys(source_urls)),
            'gold_qa_pairs': gold_qa_pairs}

all_records = [normalize_record(r, i) for i, r in enumerate(raw_dev)]
dev_df = pd.DataFrame(all_records)
dev_df = dev_df[dev_df['claim'].str.len() > 0].reset_index(drop=True)
print(f'Normalized: {len(dev_df)} records')
print(dev_df['label'].value_counts())


Dev records downloaded: 500
Normalized: 500 records
label
Refuted                 305
Supported               122
Conflicting Evidence     38
Not Enough Evidence      35
Name: count, dtype: int64


In [ ]:
# ===== Cell 6: Build test set (20-claim stratified sample, or full dev set) — CPU =====

if USE_FULL_DEV_SET:
    test_df = dev_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f'Using FULL dev set: {len(test_df)} claims (no sampling)')
else:
    test_parts = []
    for label, n in LABEL_SAMPLE.items():
        pool = dev_df[dev_df['label'] == label]
        n_sample = min(n, len(pool))
        if n_sample > 0:
            test_parts.append(pool.sample(n_sample, random_state=SEED))
        print(f'  {label:35s}: sampled {n_sample} / target {n}')
    test_df = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f'\nStratified test set: {len(test_df)} claims')

print(test_df['label'].value_counts())
display(test_df[['claim_id','claim','label']].head(5))

# Preserve the relationship between each claim and its annotated sources.
claim_source_rows = []
for _, row in test_df.iterrows():
    for qa_pair in row['gold_qa_pairs']:
        for answer in qa_pair.get('answers', []):
            source_url = answer.get('source_url', '')
            if not source_url.startswith('http'):
                continue
            claim_source_rows.append({
                'claim_id': row['claim_id'],
                'doc_id': url_to_doc_id(source_url),
                'url': source_url,
                'cached_url': answer.get('cached_source_url', ''),
            })

claim_sources_df = (
    pd.DataFrame(claim_source_rows, columns=['claim_id', 'doc_id', 'url', 'cached_url'])
    .drop_duplicates(['claim_id', 'url'])
    .reset_index(drop=True)
)
print(f'Claim-to-source links: {len(claim_sources_df)}')


  Refuted                            : sampled 8 / target 8
  Supported                          : sampled 6 / target 6
  Not Enough Evidence                : sampled 4 / target 4
  Conflicting Evidence               : sampled 2 / target 2

Stratified test set: 20 claims
label
Refuted                 8
Supported               6
Not Enough Evidence     4
Conflicting Evidence    2
Name: count, dtype: int64


,claim_id,claim,label
0,312,John Cammo was the only one to predict that Pr...,Refuted
1,233,There has been a 60% drop in government revenue.,Not Enough Evidence
2,206,There has been a 60% drop in government revenu...,Not Enough Evidence
3,271,False Facebook posts claim Philippine vice pre...,Refuted
4,98,The iPhone 12 won’t come with earphones and a ...,Supported


Claim-to-source links: 39


## STEP 2 — Download evidence web pages (CPU, needs network)

In [ ]:
# ===== Cell 7: Scrape evidence pages (round-robin URL selection, fair across claims) =====

SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT})

def fetch_html(url):
    try:
        r = SESSION.get(url, timeout=HTTP_TIMEOUT, allow_redirects=True)
        r.raise_for_status()
        ct = r.headers.get('Content-Type', '')
        if 'text/html' not in ct and 'application/xhtml' not in ct:
            return None
        return r.text
    except Exception:
        return None


def extract_text_bs(html):
    try:
        try:
            import trafilatura
            text = trafilatura.extract(html, include_comments=False, include_tables=False,
                                        favor_precision=True, no_fallback=False)
            if text and safe_word_count(text) >= MIN_WORD_COUNT:
                return clean_text(text)
        except Exception:
            pass
        soup = BeautifulSoup(html, 'lxml')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
            tag.decompose()
        return clean_text(soup.get_text(' ', strip=True))
    except Exception:
        return ''


# Round-robin: one URL from every claim before any claim gets a second,
# so evidence coverage is fair across all 20 claims rather than the first
# few claims eating the whole budget.
per_claim_urls = [list(dict.fromkeys(urls)) for urls in test_df['source_urls']]
all_unique_urls_flat = list(dict.fromkeys(u for urls in per_claim_urls for u in urls))

unique_urls, seen, round_idx = [], set(), 0
while len(unique_urls) < N_EVIDENCE_URLS:
    added = False
    for urls in per_claim_urls:
        if round_idx < len(urls) and urls[round_idx] not in seen:
            unique_urls.append(urls[round_idx]); seen.add(urls[round_idx]); added = True
            if len(unique_urls) >= N_EVIDENCE_URLS:
                break
    round_idx += 1
    if not added:
        break

print(f'Total unique URLs available: {len(all_unique_urls_flat)} -> attempting: {len(unique_urls)}')

fallback_by_url = {}
for _, source_row in claim_sources_df.iterrows():
    cached_url = clean_text(source_row['cached_url'])
    if cached_url.startswith('http'):
        fallback_by_url.setdefault(source_row['url'], cached_url)

evidence_docs = []
for url in tqdm(unique_urls, desc='Downloading evidence pages'):
    doc_id = url_to_doc_id(url)
    html = fetch_html(url)
    fetched_url = url
    if html is None and url in fallback_by_url:
        fetched_url = fallback_by_url[url]
        html = fetch_html(fetched_url)
    if html is None:
        continue
    text = extract_text_bs(html)
    wc = safe_word_count(text)
    if wc < MIN_WORD_COUNT:
        continue
    evidence_docs.append({'doc_id': doc_id, 'url': url, 'fetched_url': fetched_url,
                          'text': text, 'word_count': wc})
    time.sleep(SLEEP_SECONDS)

evidence_docs_df = pd.DataFrame(
    evidence_docs, columns=['doc_id', 'url', 'fetched_url', 'text', 'word_count']
).drop_duplicates('doc_id').reset_index(drop=True)
print(f'\nDownloaded: {len(evidence_docs_df)} evidence documents')
if len(evidence_docs_df):
    print(f'Average word count: {evidence_docs_df["word_count"].mean():.0f}')
else:
    raise RuntimeError(
        'No evidence pages were downloaded. Check network access and source availability.'
    )


Total unique URLs available: 39 -> attempting: 39



Downloaded: 30 evidence documents
Average word count: 2930


## STEP 3 — Chunk evidence documents (CPU)

In [ ]:
# ===== Cell 8: Chunking =====

def chunk_words(text, chunk_w=CHUNK_WORDS, overlap_w=OVERLAP_WORDS, min_w=MIN_CHUNK_WORDS):
    words = str(text).split()
    if not words:
        return []
    chunks = []
    start = 0
    step = chunk_w - overlap_w
    while start < len(words):
        end = min(start + chunk_w, len(words))
        part = words[start:end]
        if len(part) >= min_w or not chunks:
            chunks.append(' '.join(part))
        if end >= len(words):
            break
        start += step
    return chunks


chunk_rows = []
for _, doc in evidence_docs_df.iterrows():
    doc_id = doc['doc_id']
    text = clean_text(doc['text'])
    for ci, chunk_text in enumerate(chunk_words(text)):
        chunk_rows.append({'doc_id': doc_id, 'chunk_index': ci, 'chunk_id': f'{doc_id}_chunk_{ci:03d}',
                           'chunk_text': chunk_text, 'chunk_word_count': len(chunk_text.split()), 'url': doc['url']})

chunks_df = pd.DataFrame(chunk_rows)
chunks_df['retrieval_text'] = chunks_df['chunk_text'].str.strip()
print(f'Total chunks: {len(chunks_df)}')
if len(evidence_docs_df):
    print(f'Avg chunks/doc: {len(chunks_df)/max(len(evidence_docs_df),1):.1f}')


Total chunks: 634
Avg chunks/doc: 21.1


## SAVE — writes the files Notebook 2 needs

In [ ]:
# ===== Cell 9: Save checkpoints =====

test_df.to_parquet(CHECKPOINT_DIR / 'test_df.parquet', index=False)
chunks_df.to_parquet(CHECKPOINT_DIR / 'chunks_df.parquet', index=False)
evidence_docs_df.to_parquet(CHECKPOINT_DIR / 'evidence_docs_df.parquet', index=False)
claim_sources_df.to_parquet(CHECKPOINT_DIR / 'claim_sources_df.parquet', index=False)

print('Saved to', CHECKPOINT_DIR, ':')
print('  - test_df.parquet         (', len(test_df), 'claims )')
print('  - chunks_df.parquet       (', len(chunks_df), 'chunks )')
print('  - evidence_docs_df.parquet(', len(evidence_docs_df), 'docs )')
print('  - claim_sources_df.parquet(', len(claim_sources_df), 'links)')
print()
print('DONE. Open Notebook 2 (Retrieval Index) next.')


Saved to /content/drive/MyDrive/averitec_extended/averitec_20claim_v1/checkpoints :
  - test_df.parquet         ( 20 claims )
  - chunks_df.parquet       ( 634 chunks )
  - evidence_docs_df.parquet( 30 docs )
  - claim_sources_df.parquet( 39 links)

DONE. Open Notebook 2 (Retrieval Index) next.
